In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# Code Explainer & Reviewer — LLM-Powered Code Understanding

## 1. Project Overview

**Task:** Given a code snippet, automatically (a) explain what it does in plain English, (b) detect possible bugs or code smells, and (c) suggest concrete improvements or refactors.

**Approach:** Three specialized prompts, each tuned for a different aspect of code understanding — explanation, bug-hunting, and refactoring — so the LLM can focus on one goal at a time.

**Stack:**
- `ChatOllama` + `qwen3.5:9b` — local LLM for all analysis
- `LangChain` — message formatting
- Pure Python — no external dependencies beyond LangChain

> **No API keys required.** Everything runs locally with Ollama.

## 2. Learning Goals

| # | Skill |
|---|-------|
| 1 | **Explain code** — generate multi-level explanations (high-level → line-by-line) |
| 2 | **Detect bugs** — prompt an LLM to find logic errors, edge cases, security issues |
| 3 | **Suggest refactors** — generate idiomatic, cleaner versions of the same code |
| 4 | **Structure analysis prompts** — use system prompts to set the LLM's "reviewer persona" |
| 5 | **Parse structured output** — get JSON results from code analysis for downstream use |
| 6 | **Evaluate LLM code reasoning** — compare LLM findings against known bugs |
| 7 | **Understand LLM limitations** — see where code-understanding models succeed and fail |

## 3. Problem Statement

Developers spend a large fraction of their time **reading code**, not writing it. Studies show engineers spend 58-70% of their time understanding existing code. The challenges:

- **Onboarding** — new team members stare at unfamiliar codebases
- **Code review** — reviewers need to spot bugs and suggest improvements quickly
- **Legacy code** — poorly documented code with no original author to ask
- **Learning** — beginners can't always follow complex patterns (recursion, decorators, async)

An LLM-powered code explainer can produce instant explanations, flag likely bugs, and propose cleaner alternatives.

## 4. Why This Project Matters

- **Core NLP capability** — code is a formal language; understanding it tests structured reasoning
- **Real-world demand** — GitHub Copilot, Cursor, Cody all build on code-understanding LLMs
- **Prompt engineering practice** — code analysis requires precise, structured prompts
- **Multi-pass analysis** — explanation, bug detection, and refactoring need different prompts and personas

## 5. Pipeline Architecture

```
Code Snippet (any language)
         │
         ▼
  [Language Detection]  ── identify language for context
         │
         ├──▶ [Explanation Pass]     → plain-English walkthrough
         │          └── high-level + line-by-line
         │
         ├──▶ [Bug Detection Pass]   → potential bugs & code smells
         │          └── severity, location, fix suggestion
         │
         ├──▶ [Refactor Pass]        → improved version of the code
         │          └── with explanations of changes
         │
         └──▶ [Full Review]          → combined report
```

### Why Separate Passes?

A single "analyze this code" prompt produces shallow results. Separate passes let each prompt:
1. Focus on **one cognitive task** (explain vs critique vs rewrite)
2. Use a **different system persona** (teacher vs security auditor vs senior developer)
3. Apply **different temperatures** (0.0 for bug detection, 0.4 for refactoring)

## 6. LLMs for Code Understanding

### How LLMs Understand Code

LLMs trained on code (or general LLMs with code in their training data) can:

| Capability | How It Works | Reliability |
|------------|-------------|-------------|
| **Explain logic** | Pattern-match against millions of similar code examples | High |
| **Detect common bugs** | Recognize patterns like off-by-one, null dereference, SQL injection | Medium-High |
| **Suggest style fixes** | Know idiomatic patterns per language (PEP 8, Go fmt, etc.) | High |
| **Find edge cases** | Reason about boundary conditions and unusual inputs | Medium |
| **Verify correctness** | Formally prove code does what it claims | Low |
| **Understand context** | Know what a function does within a larger system | Low (needs context) |

### Key Insight

LLMs are **pattern matchers**, not compilers. They excel at recognizing *common* bugs and suggesting *idiomatic* improvements, but they can:
- Miss **subtle logic errors** that require deep state tracking
- **Hallucinate bugs** that don't exist
- Suggest **incorrect fixes** that introduce new bugs

Always verify LLM suggestions before applying them. Use LLM code review as a **first pass** to speed up human review, not as a replacement.

## 7. Environment Setup

In [ ]:
# Uncomment if any package is missing
# !pip install -q langchain langchain-ollama langchain-core

print("Dependencies: langchain, langchain-ollama")
print("LLM: Ollama with qwen3.5:9b (must be running locally)")

## 8. Imports

In [ ]:
import os
import re
import json
import textwrap
import warnings

os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports OK")

## 9. Configuration

In [ ]:
LLM_MODEL = "qwen3.5:9b"

TEMP_EXPLAIN = 0.2    # Explanation — mostly deterministic, clear
TEMP_BUGS = 0.0       # Bug detection — fully deterministic
TEMP_REFACTOR = 0.3   # Refactoring — slight creativity for alternatives

print("Configuration loaded")
print(f"  LLM: {LLM_MODEL}")
print(f"  Temperatures: explain={TEMP_EXPLAIN}, bugs={TEMP_BUGS}, refactor={TEMP_REFACTOR}")

## 10. Helper Functions

In [ ]:
def clean(text: str) -> str:
    """Strip thinking tags from LLM output."""
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text.strip()


def parse_json(text: str):
    """Extract JSON from LLM output."""
    text = clean(text)
    if "```" in text:
        text = re.sub(r"```(?:json)?\s*", "", text)
        text = text.replace("```", "")
    start = text.find("{")
    alt = text.find("[")
    if alt >= 0 and (start < 0 or alt < start):
        start = alt
    end = max(text.rfind("}"), text.rfind("]")) + 1
    if start >= 0 and end > start:
        try:
            return json.loads(text[start:end])
        except json.JSONDecodeError:
            pass
    return None


def ask(prompt: str, system: str = "", temperature: float = 0.2) -> str:
    """Send a prompt to the LLM and return cleaned text."""
    llm = ChatOllama(model=LLM_MODEL, temperature=temperature)
    messages = []
    if system:
        messages.append(SystemMessage(content=system))
    messages.append(HumanMessage(content=prompt))
    resp = llm.invoke(messages)
    return clean(resp.content)


def wrap_print(text: str, width: int = 90):
    """Print text with word wrapping."""
    for line in text.split("\n"):
        if line.strip():
            print(textwrap.fill(line, width=width))
        else:
            print()


# Quick LLM test
test = ask("Say 'ready'.")
print(f"LLM ready: {test[:30]}")

---

# Part A — Code Input Examples

## 11. Sample Code Snippets

We define several code snippets with **known issues** so we can evaluate how well the LLM detects them.

In [ ]:
# Snippet 1 — Python: user authentication (multiple bugs)
SNIPPET_1 = '''
import hashlib

def authenticate(username, password, db):
    query = f"SELECT password_hash FROM users WHERE username = '{username}'"
    result = db.execute(query)
    if result:
        stored_hash = result[0]
        input_hash = hashlib.md5(password.encode()).hexdigest()
        if input_hash == stored_hash:
            return True
    return False
'''.strip()

SNIPPET_1_BUGS = [
    "SQL injection via f-string interpolation",
    "MD5 is cryptographically broken for password hashing",
    "No salting of password hashes",
    "Timing attack via == comparison on hashes",
]

# Snippet 2 — Python: list processing (logic bugs)
SNIPPET_2 = '''
def remove_duplicates(items):
    seen = []
    for i in range(len(items)):
        if items[i] not in seen:
            seen.append(items[i])
        else:
            items.pop(i)
    return items

def average(numbers):
    total = 0
    for n in numbers:
        total += n
    return total / len(numbers)
'''.strip()

SNIPPET_2_BUGS = [
    "Modifying list while iterating (index shift after pop)",
    "IndexError when pop shifts remaining elements",
    "ZeroDivisionError when numbers list is empty",
    "O(n^2) complexity — using list for 'seen' instead of set",
]

# Snippet 3 — Python: file processing (resource + error handling)
SNIPPET_3 = '''
import json

def load_config(filepath):
    f = open(filepath, 'r')
    data = f.read()
    config = json.loads(data)
    return config

def save_results(results, output_path):
    f = open(output_path, 'w')
    for r in results:
        f.write(str(r) + '\\n')
    print(f"Saved {len(results)} results")
'''.strip()

SNIPPET_3_BUGS = [
    "File handle never closed (no context manager, no f.close())",
    "No exception handling for FileNotFoundError or JSONDecodeError",
    "save_results never closes the file — data may not flush",
    "Using str(r) instead of proper serialization",
]

# Snippet 4 — Python: clean code, no bugs (control case)
SNIPPET_4 = '''
from dataclasses import dataclass
from typing import Optional

@dataclass
class User:
    name: str
    email: str
    age: Optional[int] = None

    def is_adult(self) -> bool:
        return self.age is not None and self.age >= 18

    def display_name(self) -> str:
        return self.name.strip().title()
'''.strip()

SNIPPET_4_BUGS = []  # Intentionally clean

SNIPPETS = [
    ("Authentication (security bugs)", SNIPPET_1, SNIPPET_1_BUGS),
    ("List Processing (logic bugs)", SNIPPET_2, SNIPPET_2_BUGS),
    ("File I/O (resource bugs)", SNIPPET_3, SNIPPET_3_BUGS),
    ("Clean Dataclass (no bugs)", SNIPPET_4, SNIPPET_4_BUGS),
]

print(f"Loaded {len(SNIPPETS)} code snippets")
for name, code, bugs in SNIPPETS:
    lines = code.count("\n") + 1
    print(f"  • {name}: {lines} lines, {len(bugs)} known bugs")

---

# Part B — Explanation Generation

## 12. High-Level Explanation

In [ ]:
EXPLAIN_SYSTEM = """You are a patient programming teacher. You explain code to 
developers who can read code but want a clear summary of what it does and why.

Rules:
- Start with a one-sentence summary of the code's PURPOSE.
- Then explain the LOGIC step by step.
- Mention any design decisions (why this approach vs alternatives).
- Use plain English. Avoid jargon unless you define it.
- Do NOT comment on bugs or improvements — just explain what the code does."""

EXPLAIN_PROMPT = """Explain what this code does.

```python
{code}
```

Explanation:"""

print("CODE EXPLANATION — Snippet 1 (Authentication)")
print("=" * 60)
print(f"Code:\n{SNIPPET_1}")
print(f"\n{'─'*60}")

explanation_1 = ask(
    EXPLAIN_PROMPT.format(code=SNIPPET_1),
    system=EXPLAIN_SYSTEM,
    temperature=TEMP_EXPLAIN,
)

print("\nExplanation:")
wrap_print(explanation_1)

## 13. Line-by-Line Explanation

In [ ]:
LINEBY_PROMPT = """Provide a line-by-line explanation of this code.
For each significant line, explain what it does and why.

```python
{code}
```

Return ONLY JSON array:
[
  {{"line": "the code line", "explanation": "what it does"}},
  ...
]"""

print("LINE-BY-LINE — Snippet 2 (List Processing)")
print("=" * 60)
print(f"Code:\n{SNIPPET_2}")
print(f"\n{'─'*60}")

lineby_resp = ask(
    LINEBY_PROMPT.format(code=SNIPPET_2),
    system=EXPLAIN_SYSTEM,
    temperature=TEMP_EXPLAIN,
)
lines_explained = parse_json(lineby_resp)

if lines_explained and isinstance(lines_explained, list):
    for entry in lines_explained:
        line = entry.get("line", "")
        expl = entry.get("explanation", "")
        print(f"\n  {line}")
        print(f"    → {expl}")
else:
    wrap_print(lineby_resp[:500])

## 14. Multi-Level Explanation (Beginner → Expert)

In [ ]:
MULTILEVEL_PROMPT = """Explain this code at three levels of expertise.

```python
{code}
```

Return ONLY JSON:
{{
  "beginner": "Explanation for someone who just started learning Python (2-3 sentences)",
  "intermediate": "Explanation for someone who knows Python basics (3-4 sentences, mention patterns)",
  "expert": "Explanation for a senior developer (2-3 sentences, mention design trade-offs)"
}}"""

print("MULTI-LEVEL EXPLANATION — Snippet 4 (Dataclass)")
print("=" * 60)
print(f"Code:\n{SNIPPET_4}")
print(f"\n{'─'*60}")

ml_resp = ask(
    MULTILEVEL_PROMPT.format(code=SNIPPET_4),
    temperature=TEMP_EXPLAIN,
)
ml = parse_json(ml_resp)

if ml:
    for level in ["beginner", "intermediate", "expert"]:
        icon = {"beginner": "🟢", "intermediate": "🟡", "expert": "🔴"}[level]
        print(f"\n{icon} {level.upper()}")
        wrap_print(ml.get(level, ""))
else:
    wrap_print(ml_resp[:400])

---

# Part C — Bug Detection

## 15. Bug Spotting Pass

In [ ]:
BUG_SYSTEM = """You are a senior code reviewer and security auditor. Your job is to
find bugs, vulnerabilities, and code smells in the given code.

Focus on:
- Logic errors (off-by-one, wrong conditions, infinite loops)
- Security vulnerabilities (injection, broken auth, data exposure)
- Resource leaks (unclosed files, connections, memory)
- Edge cases that will crash (empty inputs, None values, division by zero)
- Performance issues (O(n^2) when O(n) is possible)

Be specific: quote the exact line that has the bug.
If the code is clean, say so — do NOT invent bugs that don't exist."""

BUG_PROMPT = """Review this code for bugs, vulnerabilities, and code smells.

```python
{code}
```

Return ONLY JSON:
{{
  "bugs": [
    {{
      "severity": "critical|high|medium|low",
      "category": "security|logic|resource|performance|style",
      "line": "the problematic code line",
      "description": "what the bug is",
      "fix": "how to fix it"
    }}
  ],
  "overall_risk": "critical|high|medium|low|clean"
}}"""

print("BUG DETECTION — Snippet 1 (Authentication)")
print("=" * 60)
print(f"Code:\n{SNIPPET_1}")
print(f"\n{'─'*60}")

bugs_resp = ask(
    BUG_PROMPT.format(code=SNIPPET_1),
    system=BUG_SYSTEM,
    temperature=TEMP_BUGS,
)
bugs_1 = parse_json(bugs_resp)

if bugs_1 and "bugs" in bugs_1:
    print(f"\nOverall Risk: {bugs_1.get('overall_risk', 'unknown').upper()}")
    print(f"Bugs Found: {len(bugs_1['bugs'])}")
    for b in bugs_1["bugs"]:
        sev = b.get("severity", "?")
        icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(sev, "⚪")
        print(f"\n  {icon} [{sev.upper()}] {b.get('category', '')}")
        print(f"    Line: {b.get('line', '')}")
        print(f"    Bug:  {b.get('description', '')}")
        print(f"    Fix:  {b.get('fix', '')}")
else:
    print("(parse error)")
    wrap_print(bugs_resp[:400])

## 16. Run Bug Detection on All Snippets

In [ ]:
print("BUG DETECTION — All Snippets")
print("=" * 60)

all_detections = []

for name, code, known_bugs in SNIPPETS:
    resp = ask(
        BUG_PROMPT.format(code=code),
        system=BUG_SYSTEM,
        temperature=TEMP_BUGS,
    )
    result = parse_json(resp)

    found = result.get("bugs", []) if result else []
    risk = result.get("overall_risk", "?") if result else "?"

    all_detections.append({
        "name": name,
        "known": known_bugs,
        "found": found,
        "risk": risk,
    })

    print(f"\n{'─'*60}")
    print(f"📋 {name}")
    print(f"   Risk: {risk.upper()} | Known bugs: {len(known_bugs)} | Found: {len(found)}")
    for b in found:
        sev = b.get("severity", "?")
        icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(sev, "⚪")
        print(f"   {icon} {b.get('description', '')[:80]}")

## 17. Evaluate Detection Accuracy

Compare LLM-detected bugs against our known bug lists.

In [ ]:
MATCH_PROMPT = """Given a list of known bugs and a list of LLM-detected bugs, determine
which known bugs were found and which were missed.

Known bugs:
{known}

LLM-detected bugs:
{found}

Return ONLY JSON:
{{
  "matched": ["known bug description that was detected"],
  "missed": ["known bug description that was NOT detected"],
  "false_positives": ["detected bug that is NOT a real bug"]
}}"""

print("DETECTION ACCURACY")
print("=" * 60)

for det in all_detections:
    if not det["known"]:  # Skip the clean snippet
        print(f"\n  📋 {det['name']}")
        fp = len(det["found"])
        print(f"     No known bugs. LLM reported {fp} issue(s).", end=" ")
        print("✅ Correct" if fp == 0 else "⚠ Possible false positives")
        continue

    known_str = "\n".join(f"- {b}" for b in det["known"])
    found_str = "\n".join(
        f"- {b.get('description', '')}" for b in det["found"]
    )

    resp = ask(
        MATCH_PROMPT.format(known=known_str, found=found_str),
        temperature=TEMP_BUGS,
    )
    match = parse_json(resp)

    if match:
        matched = match.get("matched", [])
        missed = match.get("missed", [])
        fps = match.get("false_positives", [])
        recall = len(matched) / len(det["known"]) * 100 if det["known"] else 0

        print(f"\n  📋 {det['name']}")
        print(f"     Known: {len(det['known'])} | Detected: {len(matched)} | "
              f"Missed: {len(missed)} | False pos: {len(fps)}")
        print(f"     Recall: {recall:.0f}%")
        if missed:
            for m in missed:
                print(f"     ❌ Missed: {m}")

---

# Part D — Refactor Suggestions

## 18. Refactor Pass

In [ ]:
REFACTOR_SYSTEM = """You are a senior Python developer doing a code review.
Your goal is to rewrite the code to be:
- Correct (fix any bugs)
- Idiomatic (follow Python best practices and PEP 8)
- Safe (handle edge cases and errors)
- Readable (clear variable names, proper structure)

Provide the improved code AND explain each change you made."""

REFACTOR_PROMPT = """Refactor this code. Fix any bugs, improve style, add proper
error handling where needed.

Original code:
```python
{code}
```

Return ONLY JSON:
{{
  "refactored_code": "the improved code as a string",
  "changes": [
    {{"what": "description of change", "why": "reason for the change"}}
  ]
}}"""

print("REFACTORED CODE — Snippet 1 (Authentication)")
print("=" * 60)

refactor_resp = ask(
    REFACTOR_PROMPT.format(code=SNIPPET_1),
    system=REFACTOR_SYSTEM,
    temperature=TEMP_REFACTOR,
)
refactored = parse_json(refactor_resp)

if refactored:
    print("\nImproved Code:")
    print(refactored.get("refactored_code", ""))
    print(f"\n{'─'*60}")
    print("Changes Made:")
    for c in refactored.get("changes", []):
        print(f"  ✅ {c.get('what', '')}")
        print(f"     Why: {c.get('why', '')}")
else:
    print("(parse error)")
    wrap_print(refactor_resp[:600])

## 19. Refactor Snippet 2 (List Processing)

In [ ]:
print("REFACTORED CODE — Snippet 2 (List Processing)")
print("=" * 60)
print(f"Original:\n{SNIPPET_2}")
print(f"\n{'─'*60}")

refactor_2_resp = ask(
    REFACTOR_PROMPT.format(code=SNIPPET_2),
    system=REFACTOR_SYSTEM,
    temperature=TEMP_REFACTOR,
)
refactored_2 = parse_json(refactor_2_resp)

if refactored_2:
    print("\nImproved Code:")
    print(refactored_2.get("refactored_code", ""))
    print(f"\n{'─'*60}")
    print("Changes Made:")
    for c in refactored_2.get("changes", []):
        print(f"  ✅ {c.get('what', '')}")
        print(f"     Why: {c.get('why', '')}")
else:
    wrap_print(refactor_2_resp[:600])

## 20. Refactor Snippet 3 (File I/O)

In [ ]:
print("REFACTORED CODE — Snippet 3 (File I/O)")
print("=" * 60)
print(f"Original:\n{SNIPPET_3}")
print(f"\n{'─'*60}")

refactor_3_resp = ask(
    REFACTOR_PROMPT.format(code=SNIPPET_3),
    system=REFACTOR_SYSTEM,
    temperature=TEMP_REFACTOR,
)
refactored_3 = parse_json(refactor_3_resp)

if refactored_3:
    print("\nImproved Code:")
    print(refactored_3.get("refactored_code", ""))
    print(f"\n{'─'*60}")
    print("Changes Made:")
    for c in refactored_3.get("changes", []):
        print(f"  ✅ {c.get('what', '')}")
        print(f"     Why: {c.get('why', '')}")
else:
    wrap_print(refactor_3_resp[:600])

---

# Part E — Full Code Review Pipeline

## 21. Combined Review Function

In [ ]:
def review_code(code: str, language: str = "python") -> dict:
    """
    Full code review pipeline: explain → find bugs → suggest refactor.
    Returns a structured report.
    """
    report = {"code": code, "language": language}

    # Pass 1: Explanation
    report["explanation"] = ask(
        f"Explain what this {language} code does in 3-5 sentences."
        f"\n\n```{language}\n{code}\n```",
        system=EXPLAIN_SYSTEM,
        temperature=TEMP_EXPLAIN,
    )

    # Pass 2: Bug detection
    bugs_raw = ask(
        BUG_PROMPT.format(code=code),
        system=BUG_SYSTEM,
        temperature=TEMP_BUGS,
    )
    bugs = parse_json(bugs_raw)
    report["bugs"] = bugs.get("bugs", []) if bugs else []
    report["risk"] = bugs.get("overall_risk", "unknown") if bugs else "unknown"

    # Pass 3: Refactor
    refactor_raw = ask(
        REFACTOR_PROMPT.format(code=code),
        system=REFACTOR_SYSTEM,
        temperature=TEMP_REFACTOR,
    )
    ref = parse_json(refactor_raw)
    report["refactored"] = ref.get("refactored_code", "") if ref else ""
    report["changes"] = ref.get("changes", []) if ref else []

    return report


def print_review(report: dict):
    """Pretty-print a code review report."""
    print(f"\n{'═'*60}")
    print(f"CODE REVIEW REPORT ({report.get('language', 'unknown')})")
    print(f"{'═'*60}")

    print(f"\n📖 EXPLANATION")
    print(f"{'─'*60}")
    wrap_print(report.get("explanation", ""))

    bugs = report.get("bugs", [])
    risk = report.get("risk", "unknown")
    print(f"\n🐛 BUGS FOUND: {len(bugs)} | Risk: {risk.upper()}")
    print(f"{'─'*60}")
    for b in bugs:
        sev = b.get("severity", "?")
        icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(sev, "⚪")
        print(f"  {icon} [{sev.upper()}] {b.get('description', '')}")
        print(f"    Fix: {b.get('fix', '')}")

    print(f"\n🔧 REFACTORED CODE")
    print(f"{'─'*60}")
    print(report.get("refactored", "(none)"))

    changes = report.get("changes", [])
    if changes:
        print(f"\n📝 CHANGES ({len(changes)})")
        for c in changes:
            print(f"  ✅ {c.get('what', '')}")


print("review_code() and print_review() defined")
print("Usage: report = review_code(your_code); print_review(report)")

## 22. Full Review Demo

In [ ]:
# Full review on a new, unseen snippet
NEW_SNIPPET = '''
import pickle
import os

class Cache:
    def __init__(self, cache_dir="/tmp/cache"):
        self.cache_dir = cache_dir
        os.makedirs(cache_dir, exist_ok=True)

    def get(self, key):
        path = os.path.join(self.cache_dir, key)
        if os.path.exists(path):
            with open(path, 'rb') as f:
                return pickle.load(f)
        return None

    def set(self, key, value, ttl=3600):
        path = os.path.join(self.cache_dir, key)
        with open(path, 'wb') as f:
            pickle.dump(value, f)

    def delete(self, key):
        path = os.path.join(self.cache_dir, key)
        os.remove(path)
'''.strip()

print("FULL REVIEW — Cache Class")
print(f"Code:\n{NEW_SNIPPET}")

report = review_code(NEW_SNIPPET)
print_review(report)

## 23. Review a Different Language

In [ ]:
JS_SNIPPET = '''
function fetchUserData(userId) {
  let response = fetch('/api/users/' + userId);
  let data = response.json();
  document.getElementById('name').innerHTML = data.name;
  document.getElementById('bio').innerHTML = data.bio;
  return data;
}

function deleteUser(userId) {
  if (confirm('Delete user?')) {
    fetch('/api/users/' + userId, { method: 'DELETE' });
    location.reload();
  }
}
'''.strip()

print("FULL REVIEW — JavaScript (Fetch API)")
print(f"Code:\n{JS_SNIPPET}")

js_report = review_code(JS_SNIPPET, language="javascript")
print_review(js_report)

---

## 24. Edge Case: Clean Code (False Positive Check)

A good code reviewer should **not hallucinate bugs** in clean code.

In [ ]:
print("FALSE POSITIVE TEST — Clean Dataclass")
print("=" * 60)
print(f"Code:\n{SNIPPET_4}")
print(f"\nExpected: 0 bugs (this code is clean)")
print(f"{'─'*60}")

clean_report = review_code(SNIPPET_4)

n_bugs = len(clean_report.get("bugs", []))
if n_bugs == 0:
    print(f"\n✅ PASS: LLM correctly found 0 bugs in clean code")
else:
    print(f"\n⚠ FALSE POSITIVES: LLM reported {n_bugs} bug(s) in clean code:")
    for b in clean_report["bugs"]:
        print(f"  ❌ {b.get('description', '')}")
    print("\n→ This is a known LLM limitation: tendency to find problems even when none exist")

## 25. Side-by-Side: Before vs After

In [ ]:
COMPARE_PROMPT = """Compare these two versions of the same code.
Identify what changed and rate the improvement.

ORIGINAL:
```python
{original}
```

REFACTORED:
```python
{refactored}
```

Return ONLY JSON:
{{
  "security_improvement": "1-5 (5 = major improvement)",
  "readability_improvement": "1-5",
  "correctness_improvement": "1-5",
  "summary": "2-sentence summary of the most important changes"
}}"""

# Use the authentication snippet and its refactored version
ref_code = refactored.get("refactored_code", "") if refactored else ""

if ref_code:
    compare_resp = ask(
        COMPARE_PROMPT.format(original=SNIPPET_1, refactored=ref_code),
        temperature=TEMP_BUGS,
    )
    comp = parse_json(compare_resp)

    print("BEFORE vs AFTER — Authentication")
    print("=" * 60)
    if comp:
        print(f"  Security:    {comp.get('security_improvement', '?')}/5")
        print(f"  Readability: {comp.get('readability_improvement', '?')}/5")
        print(f"  Correctness: {comp.get('correctness_improvement', '?')}/5")
        print(f"\n  Summary: {comp.get('summary', '')}")
    else:
        wrap_print(compare_resp[:300])
else:
    print("No refactored code available — re-run cell 18 first.")

## 26. Common Mistakes

| Mistake | Why It's Wrong | What to Do Instead |
|---------|---------------|-------------------|
| **"Find all bugs"** as a prompt | Too vague — LLM doesn't know what bug categories to focus on | List specific categories: security, logic, resource, performance |
| **Trusting LLM's fix blindly** | LLM may introduce new bugs while fixing old ones | Always review and test the suggested fix |
| **No false-positive testing** | LLMs tend to invent bugs in clean code | Test with known-clean code to measure false positive rate |
| **Single-pass review** | One prompt tries to explain + find bugs + refactor at once | Use separate passes with different personas and temperatures |
| **Ignoring context** | Reviewing a function without knowing its callers | Provide surrounding code or usage examples when available |
| **Not specifying language** | LLM may misinterpret syntax from one language as another | Always state the programming language explicitly |
| **High temperature for bug detection** | Randomness causes the LLM to hallucinate bugs | Use temperature 0.0 for bug detection — deterministic is safer |
| **No severity levels** | Treating all bugs as equal wastes reviewer time | Request severity (critical/high/medium/low) to prioritize fixes |

## 27. Production Improvement Ideas

1. **Git diff review** — accept a git diff as input and review only the changed lines
2. **AST-based analysis** — parse the code into an AST first, then send structured info to the LLM for deeper reasoning
3. **Multi-file context** — give the LLM the calling code so it can understand the full picture
4. **Rule-based pre-filter** — run linters (pylint, ruff) first, then ask the LLM only about issues linters can't catch
5. **CI integration** — auto-review PRs with LLM analysis posted as review comments
6. **Custom code standards** — inject team-specific coding standards into the system prompt
7. **Test case generation** — based on detected bugs, generate unit tests that would catch them
8. **Complexity scoring** — compute cyclomatic complexity and ask the LLM to simplify high-complexity functions

## 28. Exercises

### Exercise 1: Review Your Own Code

In [ ]:
# ── Exercise 1: Paste your own code ──────────────────
# Replace MY_CODE with code from your own project.

# MY_CODE = '''paste your code here'''
# report = review_code(MY_CODE, language="python")
# print_review(report)

print("Exercise 1: Paste your own code snippet and run review_code().")
print("Try code you suspect has a bug — see if the LLM finds it.")

### Exercise 2: Complexity-Aware Review

In [ ]:
# ── Exercise 2: Ask about complexity and simplification ──

COMPLEX_SNIPPET = '''
def process_orders(orders, inventory, discounts, user_tier):
    results = []
    for order in orders:
        total = 0
        for item in order['items']:
            if item['id'] in inventory:
                if inventory[item['id']] >= item['qty']:
                    price = item['price'] * item['qty']
                    if item['id'] in discounts:
                        if user_tier == 'gold':
                            price *= (1 - discounts[item['id']] * 1.5)
                        elif user_tier == 'silver':
                            price *= (1 - discounts[item['id']])
                        else:
                            price *= (1 - discounts[item['id']] * 0.5)
                    total += price
                    inventory[item['id']] -= item['qty']
                else:
                    results.append({'order': order['id'], 'error': 'out_of_stock'})
                    continue
        if total > 0:
            results.append({'order': order['id'], 'total': round(total, 2)})
    return results
'''.strip()

COMPLEXITY_PROMPT = """This code has deep nesting and is hard to read.
Refactor it to:
1. Reduce nesting (max 2 levels deep)
2. Extract helper functions where appropriate
3. Use early returns / guard clauses
4. Keep the same behavior

Code:
```python
{code}
```

Provide the refactored code and explain the structural changes:"""

print("COMPLEXITY REFACTOR — Deep Nesting")
print("=" * 60)
print(f"Original (max nesting: 6 levels):\n{COMPLEX_SNIPPET}")
print(f"\n{'─'*60}")

complexity_result = ask(
    COMPLEXITY_PROMPT.format(code=COMPLEX_SNIPPET),
    system=REFACTOR_SYSTEM,
    temperature=TEMP_REFACTOR,
)

print("\nRefactored:")
wrap_print(complexity_result)

### Exercise 3: Security-Focused Review

In [ ]:
# ── Exercise 3: Security-specific analysis ─────────────

SECURITY_SNIPPET = '''
from flask import Flask, request, render_template_string
import subprocess
import sqlite3

app = Flask(__name__)

@app.route('/search')
def search():
    query = request.args.get('q', '')
    conn = sqlite3.connect('app.db')
    results = conn.execute(f"SELECT * FROM products WHERE name LIKE '%{query}%'")
    html = f"<h1>Results for {query}</h1>"
    for row in results:
        html += f"<p>{row[1]} - ${row[2]}</p>"
    return render_template_string(html)

@app.route('/run')
def run_command():
    cmd = request.args.get('cmd', 'echo hello')
    output = subprocess.check_output(cmd, shell=True)
    return output.decode()
'''.strip()

SECURITY_SYSTEM = """You are an application security expert performing a security audit.
Focus ONLY on security vulnerabilities. For each issue, specify:
- The vulnerability type (OWASP Top 10 category if applicable)
- The exact line of code
- Severity (critical/high/medium/low)
- A concrete exploit scenario
- The secure fix"""

SECURITY_PROMPT = """Perform a security audit on this Flask application.

```python
{code}
```

Return ONLY JSON:
{{
  "vulnerabilities": [
    {{
      "type": "OWASP category",
      "severity": "critical|high|medium|low",
      "line": "the vulnerable line",
      "exploit": "how an attacker would exploit this",
      "fix": "the secure alternative"
    }}
  ],
  "risk_score": "1-10 (10 = extremely dangerous)"
}}"""

sec_resp = ask(
    SECURITY_PROMPT.format(code=SECURITY_SNIPPET),
    system=SECURITY_SYSTEM,
    temperature=TEMP_BUGS,
)
sec = parse_json(sec_resp)

print("SECURITY AUDIT — Flask App")
print("=" * 60)

if sec:
    print(f"Risk Score: {sec.get('risk_score', '?')}/10")
    for v in sec.get("vulnerabilities", []):
        sev = v.get("severity", "?")
        icon = {"critical": "🔴", "high": "🟠", "medium": "🟡", "low": "🟢"}.get(sev, "⚪")
        print(f"\n  {icon} [{sev.upper()}] {v.get('type', '')}")
        print(f"    Line:    {v.get('line', '')}")
        print(f"    Exploit: {v.get('exploit', '')}")
        print(f"    Fix:     {v.get('fix', '')}")
else:
    wrap_print(sec_resp[:500])

### Mini Challenge

1. **Bug injection test:** Take the clean `SNIPPET_4`, deliberately introduce 3 bugs (a subtle one, a medium one, and an obvious one). Run the review pipeline and see which it catches. Measure recall at each severity.

2. **Multi-language reviewer:** Feed the same algorithm (e.g., bubble sort) in Python, JavaScript, and Go. Compare: does the LLM find language-specific issues (e.g., Python's GIL, JS's type coercion, Go's error handling)?

3. **Adversarial prompting:** Write code that *looks* buggy but is actually correct (e.g., intentionally using `==` for string comparison in Python where it's valid). How many false positives does the LLM generate?

4. **Incremental review:** Take a function, review it, apply the LLM's fix, then re-review the fixed version. Does the LLM find new issues in its own fix? How many rounds until it says the code is clean?

## 29. Session Summary

In [ ]:
print("SESSION SUMMARY")
print("=" * 50)

print(f"\nCode snippets analyzed: 6")
print(f"  • 4 Python snippets (auth, lists, file I/O, dataclass)")
print(f"  • 1 JavaScript snippet (fetch API)")
print(f"  • 1 Flask app (security audit)")

print(f"\nPipeline passes:")
print(f"  ✅ Explanation — high-level, line-by-line, multi-level")
print(f"  ✅ Bug detection — with severity and category")
print(f"  ✅ Refactor suggestions — improved code + change explanations")
print(f"  ✅ Security audit — OWASP categories, exploit scenarios")

print(f"\nEvaluation:")
print(f"  ✅ Known-bug detection accuracy (recall measurement)")
print(f"  ✅ False positive test on clean code")
print(f"  ✅ Before vs after comparison scoring")

print(f"\nKey patterns:")
print(f"  • Separate prompts per task (explain / bugs / refactor)")
print(f"  • Different personas (teacher / auditor / senior dev)")
print(f"  • Temperature 0.0 for bug detection, 0.2-0.3 for generation")
print(f"  • Structured JSON output for downstream processing")

## 30. Key Takeaways

| # | Takeaway |
|---|----------|
| 1 | **Separate passes** (explain → bugs → refactor) outperform a single "analyze this code" prompt |
| 2 | **Different personas** (teacher, auditor, senior dev) produce specialized, deeper analysis |
| 3 | **Temperature 0.0** for bug detection prevents hallucinated bugs |
| 4 | **Structured JSON output** makes results machine-readable for pipelines and CI integration |
| 5 | **Always test for false positives** — LLMs tend to over-report bugs in clean code |
| 6 | **LLMs excel at common patterns** (SQL injection, resource leaks) but miss subtle logic errors |
| 7 | **Never trust refactored code blindly** — LLM fixes can introduce new bugs |
| 8 | **Severity labels** (critical/high/medium/low) help prioritize which bugs to fix first |

---

*This notebook is part of the [Machine Learning Projects](https://github.com/pypi-ahmad/Machine-Learning-Projects) repository.*